In [1]:
from datasets import load_dataset, Dataset, DatasetDict
from collections import Counter
from tqdm import tqdm
from create_ilm_examples import randomly_mask_document
import ilm.mask
from ilm.mask.util import mask_cls_str_to_type
import multiprocessing as mp
import nltk 

/home/polyester/Desktop/Programming/work/table2text/ilm/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from datasets import load_from_disk

SAVED_DATASET_DIR = "data/char_masks/wiki_bio/train"

ds_saved = load_from_disk(SAVED_DATASET_DIR)
ds_saved

Dataset({
    features: ['input_text', 'target_text'],
    num_rows: 100
})

In [3]:
ds_saved[0]["target_text"]

{'doc': 'nora reiche -lrb- born 16 september 1983 in leipzig -rrb- is a german handballer playing for thüringer hc and the german national team .\nshe won the champions league with viborg in 2009 .\nreiche made her debut on the german team in 2004 .\nshe received a bronze medal at the 2007 world championship .\n',
 'char_masks': [[[4, 52, 5]],
  [],
  [[2, 0, 136], [4, 224, 4]],
  [[3, 18, 17]],
  [[4, 110, 3]],
  [[4, 58, 2], [4, 195, 4], [3, 210, 21], [3, 267, 2]],
  [[3, 195, 33]],
  [[0, 0, 300]],
  [[4, 52, 5], [2, 188, 50], [3, 298, 1]],
  [[3, 81, 11]],
  [[4, 0, 4], [3, 52, 17], [3, 239, 12]],
  [[4, 171, 6], [1, 188, 50]],
  [[3, 130, 6]],
  [[4, 61, 1], [1, 239, 60]],
  [[3, 130, 4]],
  [[2, 0, 136], [3, 200, 3], [3, 210, 2]]]}

In [4]:
ds_saved[0]["input_text"]

{'table': {'column_header': ['nationalteam',
   'nationality',
   'article_title',
   'clubnumber',
   'position',
   'ntupdate',
   'birth_date',
   'name',
   'birth_place',
   'height',
   'currentclub',
   'image',
   'nationalyears',
   'years',
   'fullname',
   'clubs'],
  'row_number': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
  'content': ['ger',
   'german',
   'nora reiche\n',
   '9',
   'right back',
   '17 june 2009',
   '16 september 1983',
   'nora reiche',
   'leipzig , germany',
   '1.77 0',
   'thüringer hc',
   'portrait nora-reiche . jpg',
   '2004 --',
   '1994 2010 -- -- 1997 1997 -- 2007 2007 -- 2010',
   'nora reiche',
   'ssv stötteritz hc leipzig viborg hk thüringer hc']},
 'context': 'nora reiche\n'}

In [5]:
table = ds_saved[0]["input_text"]["table"]

In [6]:
alist = [c for chcnt in zip(table["column_header"], table["content"]) for c in chcnt]

In [7]:
formatted = []
for i, value in enumerate(alist):
    formatted.append("<|tab_sep|>" if i % 2 == 0 else "<|tab_eq|>")
    formatted.append(value)

formatted

['<|tab_sep|>',
 'nationalteam',
 '<|tab_eq|>',
 'ger',
 '<|tab_sep|>',
 'nationality',
 '<|tab_eq|>',
 'german',
 '<|tab_sep|>',
 'article_title',
 '<|tab_eq|>',
 'nora reiche\n',
 '<|tab_sep|>',
 'clubnumber',
 '<|tab_eq|>',
 '9',
 '<|tab_sep|>',
 'position',
 '<|tab_eq|>',
 'right back',
 '<|tab_sep|>',
 'ntupdate',
 '<|tab_eq|>',
 '17 june 2009',
 '<|tab_sep|>',
 'birth_date',
 '<|tab_eq|>',
 '16 september 1983',
 '<|tab_sep|>',
 'name',
 '<|tab_eq|>',
 'nora reiche',
 '<|tab_sep|>',
 'birth_place',
 '<|tab_eq|>',
 'leipzig , germany',
 '<|tab_sep|>',
 'height',
 '<|tab_eq|>',
 '1.77 0',
 '<|tab_sep|>',
 'currentclub',
 '<|tab_eq|>',
 'thüringer hc',
 '<|tab_sep|>',
 'image',
 '<|tab_eq|>',
 'portrait nora-reiche . jpg',
 '<|tab_sep|>',
 'nationalyears',
 '<|tab_eq|>',
 '2004 --',
 '<|tab_sep|>',
 'years',
 '<|tab_eq|>',
 '1994 2010 -- -- 1997 1997 -- 2007 2007 -- 2010',
 '<|tab_sep|>',
 'fullname',
 '<|tab_eq|>',
 'nora reiche',
 '<|tab_sep|>',
 'clubs',
 '<|tab_eq|>',
 'ssv stött

In [8]:
for row in ds_saved:
  print(row)

{'input_text': {'table': {'column_header': ['nationalteam', 'nationality', 'article_title', 'clubnumber', 'position', 'ntupdate', 'birth_date', 'name', 'birth_place', 'height', 'currentclub', 'image', 'nationalyears', 'years', 'fullname', 'clubs'], 'row_number': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'content': ['ger', 'german', 'nora reiche\n', '9', 'right back', '17 june 2009', '16 september 1983', 'nora reiche', 'leipzig , germany', '1.77 0', 'thüringer hc', 'portrait nora-reiche . jpg', '2004 --', '1994 2010 -- -- 1997 1997 -- 2007 2007 -- 2010', 'nora reiche', 'ssv stötteritz hc leipzig viborg hk thüringer hc']}, 'context': 'nora reiche\n'}, 'target_text': {'doc': 'nora reiche -lrb- born 16 september 1983 in leipzig -rrb- is a german handballer playing for thüringer hc and the german national team .\nshe won the champions league with viborg in 2009 .\nreiche made her debut on the german team in 2004 .\nshe received a bronze medal at the 2007 world championship .\n', 'ch